## Top 10 Movies

Today, we're going to build a website just like that using Flask/WTForms/SQLite/SQLAlchemy and more. It will allow us to create a beautiful website that lists our top 10 films of all time. As we watch more movies, we can always update our list and keep track of which movies to recommend people.

1, install packages

In [1]:
pip install flask flask-sqlalchemy

ERROR: Could not find a version that satisfies the requirement flask-sqlalchemy (from versions: none)
ERROR: No matching distribution found for flask-sqlalchemy


2. main 

In [ ]:
from flask import Flask, render_template

In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)
CORS(app)

app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///posts.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

db = SQLAlchemy(app)    

3. create movies data model

In [ ]:
class Movie(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(250), unique=True, nullable=False)
    year = db.Column(db.Integer, nullable=False)
    description = db.Column(db.Text, nullable=False)
    rating = db.Column(db.Float, nullable=True)
    ranking = db.Column(db.Integer, nullable=True)
    review = db.Column(db.Text, nullable=True)
    img_url = db.Column(db.String(500), nullable=False)

4. create database

In [ ]:
with app.app_context():
    db.create_all()

5. add first movie

In [ ]:
with app.app_context():
    new_movie = Movie(
        title="Phone Booth",
        year=2002,
        description="Publicist Stuart Shepard finds himself trapped in a phone booth, pinned down by an extortionist's sniper rifle. Unable to leave or receive outside help, Stuart's negotiation with the caller leads to a jaw-dropping climax.",
        rating=7.3,
        ranking=10,
        review="My favourite character was the caller.",
        img_url="https://image.tmdb.org/t/p/w500/tjrX2oWRCM3Tvarz38zlZM7Uc10.jpg"
    )

    db.session.add(new_movie)
    db.session.commit()

6. add second movie

In [ ]:
with app.app_context():
    second_movie = Movie(
        title="Avatar The Way of Water",
        year=2022,
        description="Set more than a decade after the events of the first film, learn the story of the Sully family (Jake, Neytiri, and their kids), the trouble that follows them

Requiremrt 1 - GET Blog post

In [ ]:
@app.route("/posts", methods=["GET"])
def get_posts():
    posts = db.session.execute(
        db.select(BlogPost)
    ).scalars().all()

    return jsonify(posts=[post.to_dict() for post in posts])

Get one blog post

In [ ]:
@app.route("/posts/<int:post_id>", methods=["GET"])
def get_post(post_id):
    post = db.get_or_404(BlogPost, post_id)

    return jsonify(post=post.to_dict())

Requirement 2 — POST a new blog post

In [ ]:
@app.route("/posts", methods=["POST"])
def add_post():
    data = request.get_json()

    new_post = BlogPost(
        title=data["title"],
        subtitle=data["subtitle"],
        body=data["body"],
        author=data["author"],
        date=data["date"],
        img_url=data["img_url"]
    )

    db.session.add(new_post)
    db.session.commit()

    return jsonify(message="Blog post created successfully", post=new_post.to_dict()), 201

Requirement 3 — Edit an existing post

In [ ]:
@app.route("/posts/<int:post_id>", methods=["PUT"])
def edit_post(post_id):
    post = db.get_or_404(BlogPost, post_id)

    data = request.get_json()

    post.title = data.get("title", post.title)
    post.subtitle = data.get("subtitle", post.subtitle)
    post.body = data.get("body", post.body)
    post.author = data.get("author", post.author)
    post.date = data.get("date", post.date)
    post.img_url = data.get("img_url", post.img_url)

    db.session.commit()

    return jsonify(
        message="Blog post updated successfully",
        post=post.to_dict()
    )

Requirement 4 — DELETE a blog post

In [ ]:
@app.route("/posts/<int:post_id>", methods=["DELETE"])
def delete_post(post_id):
    post = db.get_or_404(BlogPost, post_id)

    db.session.delete(post)
    db.session.commit()

    return jsonify(
        message="Blog post deleted successfully"
    )